In [7]:
import numpy as np
import matplotlib.pyplot as plt
import lib_debug

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# -------------------- Synthetic function validation and 3D projection plots --------------------
# The three custom 2D synthetic functions are implemented in lib_debug.py.
# This cell checks that each function returns values from the requested formula
# for the same input points, then visualizes each surface with a 3D projection.


def _expected_f1_sharp_vs_broad(x):
    x = np.asarray(x, dtype=float)

    a = np.array([0.30, 0.30])
    b = np.array([0.72, 0.72])

    s_a = 0.035
    s_b = 0.13

    peak_a = 1.25 * np.exp(-np.sum((x - a) ** 2, axis=-1) / (2.0 * s_a**2))
    peak_b = 1.00 * np.exp(-np.sum((x - b) ** 2, axis=-1) / (2.0 * s_b**2))

    return peak_a + peak_b


def _expected_f2_boundary_risk(x, out_of_bounds_value=None):
    x = np.asarray(x, dtype=float)

    edge_center = np.array([0.95, 0.50])
    inner_center = np.array([0.68, 0.50])

    s1 = 0.035
    s2 = 0.12
    s_c = 0.15

    edge = 1.3 * np.exp(
        -((x[..., 0] - edge_center[0]) ** 2) / (2.0 * s1**2)
        -((x[..., 1] - edge_center[1]) ** 2) / (2.0 * s2**2)
    )

    inner = 1.0 * np.exp(
        -np.sum((x - inner_center) ** 2, axis=-1) / (2.0 * s_c**2)
    )

    y = edge + inner

    if out_of_bounds_value is not None:
        in_bounds = np.all((0.0 <= x) & (x <= 1.0), axis=-1)
        y = np.where(in_bounds, y, out_of_bounds_value)

    return y


def _expected_f3_oscillatory_fragility(x):
    x = np.asarray(x, dtype=float)

    s_x = 0.06
    s_r = 0.18

    fragile = (
        1.15
        * np.exp(-((x[..., 0] - 0.35) ** 2) / (2.0 * s_x**2))
        * (0.75 + 0.25 * np.cos(24.0 * np.pi * (x[..., 1] - 0.50)))
    )

    robust = 0.95 * np.exp(
        -((x[..., 0] - 0.72) ** 2) / (2.0 * s_r**2)
        -((x[..., 1] - 0.50) ** 2) / (2.0 * s_r**2)
    )

    return fragile + robust


_validation_points = np.array(
    [
        [0.00, 0.00],
        [0.30, 0.30],
        [0.35, 0.50],
        [0.68, 0.50],
        [0.72, 0.72],
        [0.95, 0.50],
        [1.00, 1.00],
    ]
)
_out_of_bounds_points = np.array(
    [
        [-0.10, 0.50],
        [0.50, 1.10],
        [0.95, 0.50],
    ]
)

np.testing.assert_allclose(
    lib_debug.f1_sharp_vs_broad(_validation_points),
    _expected_f1_sharp_vs_broad(_validation_points),
)
np.testing.assert_allclose(
    lib_debug.f2_boundary_risk(_validation_points),
    _expected_f2_boundary_risk(_validation_points),
)
np.testing.assert_allclose(
    lib_debug.f2_boundary_risk(_out_of_bounds_points, out_of_bounds_value=-1.0),
    _expected_f2_boundary_risk(_out_of_bounds_points, out_of_bounds_value=-1.0),
)
np.testing.assert_allclose(
    lib_debug.f3_oscillatory_fragility(_validation_points),
    _expected_f3_oscillatory_fragility(_validation_points),
)
print("Custom synthetic functions match the requested formulas for the validation inputs.")

_grid_axis = np.linspace(0.0, 1.0, 120)
_X1, _X2 = np.meshgrid(_grid_axis, _grid_axis)
_grid_points = np.column_stack([_X1.ravel(), _X2.ravel()])
_surfaces = [
    ("f1_sharp_vs_broad", lib_debug.f1_sharp_vs_broad(_grid_points).reshape(_X1.shape)),
    ("f2_boundary_risk", lib_debug.f2_boundary_risk(_grid_points).reshape(_X1.shape)),
    ("f3_oscillatory_fragility", lib_debug.f3_oscillatory_fragility(_grid_points).reshape(_X1.shape)),
]

fig = plt.figure(figsize=(18, 5))
for idx, (title, values) in enumerate(_surfaces, start=1):
    ax = fig.add_subplot(1, 3, idx, projection="3d")
    surf = ax.plot_surface(_X1, _X2, values, cmap="viridis", linewidth=0, antialiased=True)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_zlabel("f(x)")
    ax.view_init(elev=30, azim=-135)
    fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.08)

fig.suptitle("Custom 2D synthetic functions: 3D projections", y=1.02)
plt.tight_layout()
plt.show()


In [8]:
# -------------------- robust LCB on J(x) Bayesian Optimization --------------------
# This notebook is intentionally minimal: it only runs robust LCB on J(x).
# sEI / DGSM / StableOpt / AIRBO / uGP-UCB / USeMO / Unscented BO / CVaR /
# mean-variance robust objectives are not used here.

# -------------------- 1. Synthetic function switch --------------------
# Switch the synthetic function by comment-in / comment-out.
# Keep the variable names below unchanged when changing problems.

#function_name = "branin"
#d = 2
#gamma = 0.02

# function_name = "hartmann6"
# d = 6
# gamma = 1.0

# function_name = "f1_sharp_vs_broad"
# d = 2
# gamma = 50.0

# function_name = "f2_boundary_risk"
# d = 2
# gamma = 50.0

# function_name = "f3_oscillatory_fragility"
# d = 2
# gamma = 50.0

function_name = "ackley"
d = 2
gamma = 0.05

f_true, bounds, Sigma, canonical_name = lib_debug.get_synthetic_problem(
    function_name=function_name,
    d=d,
)
lower_bounds = bounds[:, 0]
upper_bounds = bounds[:, 1]

# -------------------- 2. Default experiment settings --------------------
random_seed = 0
rng = np.random.default_rng(random_seed)
noise_std = 0.0
noise_var = noise_std ** 2
n_initial = 5 * d
n_iter = 30
n_perturb_acq = 64
n_perturb_validation = 512
kappa = 2.0
n_candidates = 2000 if d <= 2 else 5000

STRATEGY = "robust_lcb_J"
print(f"--- Strategy Selected: {STRATEGY} ---")
print(f"--- Function Selected: {canonical_name} (d={d}) ---")

# -------------------- 3. Initialization --------------------
X_train = rng.uniform(lower_bounds, upper_bounds, size=(n_initial, d))
y_train = f_true(X_train).reshape(-1, 1)
if noise_std > 0:
    y_train = y_train + rng.normal(0.0, noise_std, size=y_train.shape)

# -------------------- 4. Bayesian Optimization Loop --------------------
print(f"--- Starting Optimization Loop ({n_iter} iterations) ---")
print(f"{'Iter':<5} | {'Best y':<12} | {'New y':<12} | {'robust LCB J':<14}")
print("-" * 60)

x_robust_lcb_J = None
for i in range(n_iter):
    # 1. Fit GP to the current nominal observations D={(X, y)}.
    gp = lib_debug.KernelGPRegressor(
        kernel=lib_debug.rbf_kernel,
        gamma=gamma,
        noise_var=noise_var,
    ).fit(X_train, y_train)

    # 2. Minimize alpha_rLCB(x)=mu_J(x)-kappa*sigma_J(x).
    #    The optimizer uses common random perturbation deltas within this BO iteration.
    x_next, robust_lcb_value = lib_debug.optimize_robust_lcb_by_random_search(
        gp,
        Sigma,
        bounds,
        rng,
        n_perturb=n_perturb_acq,
        kappa=kappa,
        n_candidates=n_candidates,
    )

    # 3. Observe the nominal black-box function once at x_next.
    #    We do not observe f_true(x_next + delta) in this PoC.
    y_next = f_true(x_next.reshape(1, -1)).reshape(1, 1)
    if noise_std > 0:
        y_next = y_next + rng.normal(0.0, noise_std, size=(1, 1))

    # 4. Update data.
    X_train = np.vstack([X_train, x_next])
    y_train = np.vstack([y_train, y_next])
    x_robust_lcb_J = x_next

    print(
        f"{i + 1:<5} | {np.min(y_train):<12.6f} | "
        f"{y_next.item():<12.6f} | {robust_lcb_value:<14.6f}"
    )

# -------------------- 5. Final surrogate robust mean validation --------------------
gp = lib_debug.KernelGPRegressor(
    kernel=lib_debug.rbf_kernel,
    gamma=gamma,
    noise_var=noise_var,
).fit(X_train, y_train)

best_idx_final = int(np.argmin(y_train))
best_observed_x = X_train[best_idx_final]
best_observed_y = float(y_train[best_idx_final, 0])

validation = lib_debug.validate_surrogate_robust_mean(
    {
        "best_observed": best_observed_x,
        "robust_lcb_J": x_robust_lcb_J,
    },
    gp,
    Sigma,
    bounds,
    n_mc=n_perturb_validation,
    rng=rng,
    use_cov=True,
)
robust_best = min(validation, key=lambda name: validation[name]["posterior_robust_mean"])

# -------------------- 6. Required summaries --------------------
print("\nBO summary")
print(f"function name: {canonical_name}")
print(f"dimension d: {d}")
print(f"bounds:\n{bounds}")
print(f"n_initial: {n_initial}")
print(f"n_iter: {n_iter}")
print(f"n_perturb_acq: {n_perturb_acq}")
print(f"kappa: {kappa}")
print(f"Sigma:\n{Sigma}")
print(f"best observed y: {best_observed_y:.6f}")
print(f"best observed x: {best_observed_x}")
print(f"final robust_lcb_J recommended x: {x_robust_lcb_J}")

print("\nvalidation summary")
header = (
    f"{'candidate':<18} {'nominal_posterior_mean':>24} "
    f"{'nominal_posterior_std':>23} {'posterior_robust_mean':>24} "
    f"{'posterior_robust_std':>23}"
)
print(header)
print("-" * len(header))
for candidate, stats in validation.items():
    print(
        f"{candidate:<18} {stats['nominal_posterior_mean']:>24.6f} "
        f"{stats['nominal_posterior_std']:>23.6f} "
        f"{stats['posterior_robust_mean']:>24.6f} "
        f"{stats['posterior_robust_std']:>23.6f}"
    )
print(f"\nrobust best by posterior_robust_mean: {robust_best}")



--- Strategy Selected: robust_lcb_J ---
--- Function Selected: Ackley (d=6) ---
--- Starting Optimization Loop (30 iterations) ---
Iter  | Best y       | New y        | robust LCB J  
------------------------------------------------------------
1     | 8.222388     | 13.971551    | -1.777175     
2     | 8.222388     | 11.629590    | -1.608090     
3     | 8.222388     | 12.461407    | -1.310548     
4     | 8.222388     | 13.375179    | -1.545528     
5     | 8.222388     | 12.639427    | -0.964340     
6     | 8.222388     | 13.329911    | -1.480719     
7     | 8.222388     | 12.709722    | -0.900926     
8     | 8.222388     | 13.098256    | -0.967808     
9     | 8.222388     | 12.916926    | -0.789733     
10    | 8.222388     | 12.893356    | -0.892288     
11    | 8.222388     | 12.779370    | -0.701666     
12    | 8.222388     | 13.236844    | -0.730827     
13    | 8.222388     | 13.246376    | -0.566125     
14    | 8.222388     | 12.686045    | -0.611233     
15    | 8.222